# dashboard_manifest_required_v1

目标：验证 dashboard 默认只接受 `analysis_manifest.json` 或 `artifact_manifest.json`，旧 fixed sidecar 只能通过显式 `allow_legacy_sidecars=True` 作为迁移检查使用。

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate))
        break

from src.analysis import backfill_analysis_manifest
from src.dashboard import load_model_matrix_dashboard, write_model_matrix_dashboard_export

CARD = REPO_ROOT / 'research/dashboard_manifest_required_v1'
LEGACY_CARD = CARD / 'legacy_card'
for path in (LEGACY_CARD,):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


In [2]:
(LEGACY_CARD / 'analysis_summary.json').write_text(
    json.dumps(
        {
            'schema_version': 'analysis.report.v1',
            'status': 'legacy_fixed_sidecars_completed',
            'start': '2026-01-01',
            'end': '2026-06-09',
            'model_count': 1,
            'note': '中文 review：无 manifest 的旧 dashboard card。',
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)
pd.DataFrame(
    [{'rank': 1, 'model': 'ridge_mse', 'return_pct': 2.5, 'relative_return_pct': 1.0, 'stability_score': 3.0}]
).to_csv(LEGACY_CARD / 'model_comparison.csv', index=False)
pd.DataFrame([{'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 2.5}]).to_csv(
    LEGACY_CARD / 'regime_metrics.csv', index=False
)
pd.DataFrame([{'model': 'ridge_mse', 'window': 'test', 'return_pct': 2.5}]).to_csv(
    LEGACY_CARD / 'window_metrics.csv', index=False
)
pd.DataFrame([{'model': 'ridge_mse', 'prediction': 0.1}]).to_csv(LEGACY_CARD / 'predictions.csv', index=False)
pd.DataFrame([{'model': 'ridge_mse', 'nav': 1.0}]).to_csv(LEGACY_CARD / 'daily_backtest.csv', index=False)
sorted(path.name for path in LEGACY_CARD.iterdir())


['analysis_summary.json',
 'daily_backtest.csv',
 'model_comparison.csv',
 'predictions.csv',
 'regime_metrics.csv',
 'window_metrics.csv']

In [3]:
default_error = ''
try:
    load_model_matrix_dashboard(LEGACY_CARD)
except FileNotFoundError as exc:
    default_error = str(exc)

assert 'analysis_manifest.json or artifact_manifest.json is required' in default_error

legacy_payload = load_model_matrix_dashboard(LEGACY_CARD, allow_legacy_sidecars=True)
assert legacy_payload['manifest_source']['kind'] == 'fixed_sidecars'
assert legacy_payload['top_models'][0]['model'] == 'ridge_mse'
assert 'predictions' not in legacy_payload['artifacts']
default_error


'analysis_manifest.json or artifact_manifest.json is required for dashboard load: /Users/rziiiii/dev/workspace/research/dashboard_manifest_required_v1/legacy_card. Run analysis manifest backfill or pass allow_legacy_sidecars=True for an explicit migration check.'

In [4]:
manifest = backfill_analysis_manifest(
    LEGACY_CARD,
    dataset_name='dashboard_manifest_required_v1',
    universe=('SOXL',),
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(LEGACY_CARD),),
    overwrite=True,
)
manifest_payload = load_model_matrix_dashboard(LEGACY_CARD)
export_payload = write_model_matrix_dashboard_export(
    LEGACY_CARD,
    output_path=CARD / 'dashboard_export.json',
    top_n=3,
)

assert manifest_payload['manifest_source']['kind'] == 'analysis_manifest'
assert manifest_payload['manifest_source']['manifest_hash'] == manifest['manifest_hash']
assert export_payload['manifest_source']['kind'] == 'analysis_manifest'
assert 'predictions' not in manifest['paths']
assert 'daily_backtest' not in manifest['paths']
manifest_payload['manifest_source']


{'kind': 'analysis_manifest',
 'path': '/Users/rziiiii/dev/workspace/research/dashboard_manifest_required_v1/legacy_card/analysis_manifest.json',
 'schema_version': 'artifact.v1',
 'manifest_hash': '338551326b83c5682e0bbd2f68d9783ce6aa64b011a30fe940a8d66774d18471',
 'producer': 'src.analysis.manifest_backfill'}

In [5]:
summary = {
    'schema_version': 'research.dashboard_manifest_required_v1.summary.v1',
    'status': 'completed',
    'artifact_id': 'dashboard_manifest_required_v1',
    'default_requires_manifest': bool(default_error),
    'default_error_contains_manifest_requirement': 'analysis_manifest.json or artifact_manifest.json is required' in default_error,
    'legacy_explicit_manifest_kind': legacy_payload['manifest_source']['kind'],
    'backfilled_manifest_kind': manifest_payload['manifest_source']['kind'],
    'backfilled_manifest_hash': manifest['manifest_hash'],
    'private_paths_exposed_in_manifest': sorted(
        key for key in manifest['paths'] if key in {'predictions', 'daily_backtest', 'adapter_predictions'}
    ),
    'dashboard_export_manifest_kind': export_payload['manifest_source']['kind'],
    'conclusion': '中文结论：dashboard 默认拒绝无 manifest 的 fixed sidecar；legacy 读取只能显式用于迁移检查，正式 export 走 backfilled manifest。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.dashboard_manifest_required_v1.summary.v1',
 'status': 'completed',
 'artifact_id': 'dashboard_manifest_required_v1',
 'default_requires_manifest': True,
 'default_error_contains_manifest_requirement': True,
 'legacy_explicit_manifest_kind': 'fixed_sidecars',
 'backfilled_manifest_kind': 'analysis_manifest',
 'backfilled_manifest_hash': '338551326b83c5682e0bbd2f68d9783ce6aa64b011a30fe940a8d66774d18471',
 'private_paths_exposed_in_manifest': [],
 'dashboard_export_manifest_kind': 'analysis_manifest',
 'conclusion': '中文结论：dashboard 默认拒绝无 manifest 的 fixed sidecar；legacy 读取只能显式用于迁移检查，正式 export 走 backfilled manifest。'}

## 中文结论

- 做了什么：验证 dashboard 默认要求 `analysis_manifest.json` 或 `artifact_manifest.json`，无 manifest 的旧 fixed sidecar 只能显式传 `allow_legacy_sidecars=True` 读取。
- 核心指标：默认读取旧卡片会报 manifest requirement；显式 legacy 模式返回 `fixed_sidecars`；backfill 后默认读取返回 `analysis_manifest` 并可正常写 `dashboard_export.json`。
- regime 差异：示例只覆盖 `bull` regime 的最小卡片，目标是验证框架读取边界，不重新评估真实 regime 表现。
- 是否晋升：不晋升策略；晋升 dashboard 读取契约，减少默认路径绕过 manifest 的机会。
- 风险：历史 notebook 若直接读取无 manifest card，需要迁移为 backfilled manifest 或显式声明 legacy 迁移模式。
- 下一步：继续把旧 research card 批量 backfill 到 manifest-first，并让 schedule 的 dashboard refresh 默认打开 preflight/backfill。